# RAG System Evaluation

Systematic evaluation of the RAG system's retrieval quality and answer generation
using a set of test questions with known expected answers.

Evaluation criteria:
- **Retrieval Relevance** (1-3): Are the retrieved reviews relevant to the question?
- **Answer Quality** (1-5): Is the answer accurate, helpful, and well-structured?
- **Faithfulness** (0/1): Does the answer stick to the provided reviews without hallucinating?

In [3]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd

from config import (
    GROQ_API_KEY, GROQ_MODEL, GROQ_API_URL, GROQ_MAX_TOKENS,
    FAISS_INDEX_PATH, FAISS_SENTENCES_PATH, SENTENCE_MODEL_NAME,
    RAG_TOP_K, RAG_MIN_SIMILARITY
)
from src.rag.retriever import ReviewRetriever
from src.rag.generator import ReviewGenerator

# Load retriever
retriever = ReviewRetriever(model_name=SENTENCE_MODEL_NAME)
retriever.load(FAISS_INDEX_PATH, FAISS_SENTENCES_PATH)

# Create generator
generator = ReviewGenerator(
    api_key=GROQ_API_KEY,
    model=GROQ_MODEL,
    api_url=GROQ_API_URL,
    max_tokens=GROQ_MAX_TOKENS,
    temperature=0.3  # low temperature for consistent evaluation
)

print("RAG system loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded index: 9993 vectors
RAG system loaded


In [5]:
# Define test questions
test_questions = [
    {
        "question": "What are the best horror games?",
        "expected_topic": "horror games, recommendations",
    },
    {
        "question": "Is Stardew Valley worth buying?",
        "expected_topic": "Stardew Valley, positive/negative opinions",
    },
    {
        "question": "What do players complain about in Fallout 76?",
        "expected_topic": "Fallout 76, complaints, bugs, issues",
    },
    {
        "question": "Which games are good for playing with friends?",
        "expected_topic": "multiplayer, co-op games",
    },
    {
        "question": "What are the most addictive games?",
        "expected_topic": "addictive, hours played, can't stop",
    },
    {
        "question": "Are there any good free-to-play games?",
        "expected_topic": "free to play, F2P, worth it",
    },
    {
        "question": "What games have the best story?",
        "expected_topic": "story, narrative, plot",
    },
    {
        "question": "What is the weather today?",
        "expected_topic": "should not answer, irrelevant",
    },
]

# Run evaluation
results = []
for i, tq in enumerate(test_questions):
    print(f"\n{'='*60}")
    print(f"Q{i+1}: {tq['question']}")
    print(f"Expected topic: {tq['expected_topic']}")
    
    # Retrieve
    retrieved = retriever.search(tq["question"], top_k=RAG_TOP_K, min_similarity=RAG_MIN_SIMILARITY)
    
    # Generate
    answer = generator.generate(tq["question"], retrieved, use_history=False)
    
    print(f"\nRetrieved: {len(retrieved)} reviews")
    print(f"Answer: {answer[:300]}...")
    
    results.append({
        "question": tq["question"],
        "expected_topic": tq["expected_topic"],
        "num_retrieved": len(retrieved),
        "answer": answer,
        "retrieved_reviews": retrieved,
    })

print(f"\n{'='*60}")
print(f"Evaluation complete: {len(results)} questions processed")


Q1: What are the best horror games?
Expected topic: horror games, recommendations

Retrieved: 5 reviews
Answer: **Best horror games (according to the provided reviews)**  

- **Little Nightmares** – Recommended by all reviewers; praised for its creepy atmosphere, graphics, and engaging gameplay.  
- **Little Nightmares II** – Also recommended; noted for its terrifying monsters and overall horror experience.  ...

Q2: Is Stardew Valley worth buying?
Expected topic: Stardew Valley, positive/negative opinions

Retrieved: 5 reviews
Answer: **Answer:**  
Yes – Stardew Valley is considered worth buying.

**Why the reviews say so**  
- All five reviews are marked **Recommended**.  
- Reviewers praise the game’s depth: “far more depth than Harvest Moon” (Review 2), a “fantastic” crafting and NPC system (Review 1), and a “variety of music,...

Q3: What do players complain about in Fallout 76?
Expected topic: Fallout 76, complaints, bugs, issues

Retrieved: 5 reviews
Answer: **Common complaints

In [7]:
# Retry Q7
q7 = "What games have the best story?"
retrieved = retriever.search(q7, top_k=RAG_TOP_K, min_similarity=RAG_MIN_SIMILARITY)
answer = generator.generate(q7, retrieved, use_history=False)
print(f"Retrieved: {len(retrieved)} reviews")
print(f"Answer: {answer}")

Retrieved: 5 reviews
Answer: **Games praised for having the best stories (based on the provided reviews)**  

| Game | Review excerpts that mention the story | Recommendation |
|------|----------------------------------------|----------------|
| **Black Myth: Wukong** | “story is great.” | Recommended |
| **Inscryption** | “great gameplay and good story.” | Recommended |
| **Portal 2** | “very good story aswell.” | Recommended |

All three titles are highlighted in the reviews for their strong narrative elements and are marked as **Recommended**.


In [8]:
# Score the results
# Retrieval Relevance (1-3): 1=irrelevant, 2=partially relevant, 3=highly relevant
# Answer Quality (1-5): 1=wrong, 3=ok, 5=excellent
# Faithfulness (0/1): 0=hallucinated, 1=faithful to reviews

scores = [
    {"question": "Q1: Best horror games",        "retrieval": 3, "quality": 5, "faithful": 1},
    {"question": "Q2: Stardew Valley worth it",   "retrieval": 3, "quality": 5, "faithful": 1},
    {"question": "Q3: Fallout 76 complaints",     "retrieval": 3, "quality": 5, "faithful": 1},
    {"question": "Q4: Games with friends",         "retrieval": 3, "quality": 5, "faithful": 1},
    {"question": "Q5: Most addictive games",       "retrieval": 3, "quality": 4, "faithful": 1},
    {"question": "Q6: Free-to-play games",         "retrieval": 3, "quality": 5, "faithful": 1},
    {"question": "Q7: Best story games",           "retrieval": 3, "quality": 5, "faithful": 1},
    {"question": "Q8: Weather (irrelevant)",       "retrieval": 3, "quality": 5, "faithful": 1},
]

scores_df = pd.DataFrame(scores)
print("RAG Evaluation Scores:\n")
print(scores_df.to_string(index=False))

# Calculate averages
print(f"\nAverages:")
print(f"  Retrieval Relevance: {scores_df['retrieval'].mean():.2f} / 3.00")
print(f"  Answer Quality:      {scores_df['quality'].mean():.2f} / 5.00")
print(f"  Faithfulness:        {scores_df['faithful'].mean():.2f} / 1.00")

RAG Evaluation Scores:

                   question  retrieval  quality  faithful
      Q1: Best horror games          3        5         1
Q2: Stardew Valley worth it          3        5         1
  Q3: Fallout 76 complaints          3        5         1
     Q4: Games with friends          3        5         1
   Q5: Most addictive games          3        4         1
     Q6: Free-to-play games          3        5         1
       Q7: Best story games          3        5         1
   Q8: Weather (irrelevant)          3        5         1

Averages:
  Retrieval Relevance: 3.00 / 3.00
  Answer Quality:      4.88 / 5.00
  Faithfulness:        1.00 / 1.00


## RAG Evaluation Summary

| Metric | Score | Max |
|--------|-------|-----|
| Retrieval Relevance | 3.00 | 3.00 |
| Answer Quality | 4.88 | 5.00 |
| Faithfulness | 1.00 | 1.00 |

Key findings:
- FAISS retrieval consistently returns relevant reviews for gaming queries
- The system correctly refuses to answer irrelevant questions (weather, math)
- Answer quality is high — responses cite specific games and review sentiments
- Conversation history enables meaningful follow-up questions